In [1]:
# =============================================================================
# H&M recsys — end to end in one cell
#   load -> candidates (S1..S6) -> features -> LGBMRanker -> MAP@12
#
# Design (matches the train/valid/test discipline):
#   TRAIN_WEEK  2020-36 : ranker is trained on these labels
#   VALID_WEEK  2020-37 : hyperparameter selection happens here
#   TEST_WEEK   2020-38 : touched ONCE, at the very end
# Candidates for week W are always built from transactions strictly before W.
# =============================================================================
import os, gc, time
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import lightgbm as lgb

t_start = time.time()

# ----------------------------- config ----------------------------------------
DATA_DIR = "/home/vaichauhan/nfs/Projects/recommendation"

TRAIN_WEEK = "2020-35"
VALID_WEEK = "2020-36"
TEST_WEEK  = "2020-37"

RECENT_WEEKS  = 8
TOPK_RECENT   = 30
TOPK_SEASONAL = 30
SEASONAL_SPAN = 2
REPURCHASE_N  = 30
I2I_WEEKS     = 8
I2I_TOPK      = 20
I2I_MIN_SALES = 5
I2I_SEEDS     = 5
IDX_TOP_N     = 30

USE_CF     = True          # flip to False if implicit misbehaves
CF_WINDOWS = (8, 52)       # locked recipe: two ALS models, merged as separate sources
CF_FACTORS = 64
CF_ITERS   = 15
CF_REG     = 0.01
CF_TOPK    = 30

CANDIDATE_CAP = None       # None = no truncation (matches notebook merge_sources)
FINAL_K       = 12
SEED          = 42
TRAIN_CUST_SAMPLE = 120_000   # cap training customers to control memory; None = all

rng = np.random.default_rng(SEED)


# ----------------------------- load ------------------------------------------
def load_data():
    tr = pd.read_csv(os.path.join(DATA_DIR, "transactions_train.csv"),
                     dtype={'article_id': str})
    cu = pd.read_csv(os.path.join(DATA_DIR, "customers.csv"),
                     dtype={'customer_id': str})
    ar = pd.read_csv(os.path.join(DATA_DIR, "articles.csv"),
                     dtype={'article_id': str, 'product_code': str})
    tr['t_dat'] = pd.to_datetime(tr['t_dat'])
    tr['week'] = tr['t_dat'].dt.strftime('%Y-%U')
    return tr, cu, ar


print("loading...")
tx, customers, articles = load_data()
print(f"  tx {tx.shape} | customers {customers.shape} | articles {articles.shape}")

ART_META = articles[['article_id', 'colour_group_name', 'index_group_name',
                     'product_type_name', 'department_name',
                     'garment_group_name']].copy()

CAT_COLS = ['product_type_name', 'department_name', 'garment_group_name',
            'colour_group_name', 'index_group_name',
            'club_member_status', 'fashion_news_frequency']


# --------------------------- candidate sources -------------------------------
def s1_recent_bestsellers(hist, cust_list, k=RECENT_WEEKS, topn=TOPK_RECENT):
    weeks = sorted(hist['week'].unique())[-k:]
    top = (hist[hist['week'].isin(weeks)]['article_id']
           .value_counts().head(topn).index.tolist())
    return {c: list(top) for c in cust_list}


def s2_seasonal_bestsellers(hist, cust_list, week_number,
                            span=SEASONAL_SPAN, topn=TOPK_SEASONAL):
    yr = hist['t_dat'].dt.year.max()
    wanted = [f"{y}-{w:02d}" for y in (yr - 1, yr - 2)
              for w in range(week_number - span, week_number + span + 1)]
    avail = set(hist['week'].unique())
    wanted = [w for w in wanted if w in avail]
    top = (hist[hist['week'].isin(wanted)]['article_id']
           .value_counts().head(topn).index.tolist())
    return {c: list(top) for c in cust_list}


def _recent_distinct(hist, n):
    h = hist.sort_values('t_dat', ascending=False)
    return (h.groupby('customer_id')['article_id']
            .apply(lambda s: list(dict.fromkeys(s.tolist()))[:n])
            .to_dict())


def s3_repurchase(hist, n=REPURCHASE_N):
    return _recent_distinct(hist, n)


def build_i2i_map(hist, k=I2I_WEEKS, min_sales=I2I_MIN_SALES, topk=I2I_TOPK):
    weeks = sorted(hist['week'].unique())[-k:]
    d = hist[hist['week'].isin(weeks)]
    vc = d['article_id'].value_counts()
    d = d[d['article_id'].isin(set(vc[vc >= min_sales].index))]

    ccodes, _ = pd.factorize(d['customer_id'])
    acodes, auniq = pd.factorize(d['article_id'])
    M = csr_matrix((np.ones(len(d), np.float32), (ccodes, acodes)),
                   shape=(ccodes.max() + 1, len(auniq)))
    M.data[:] = 1.0
    C = (M.T @ M).tocsr()

    i2i = {}
    for a in range(C.shape[0]):
        s, e = C.indptr[a], C.indptr[a + 1]
        cols, vals = C.indices[s:e], C.data[s:e]
        m = cols != a
        cols, vals = cols[m], vals[m]
        if len(cols) == 0:
            continue
        if len(cols) > topk:
            sel = np.argpartition(-vals, topk)[:topk]
            cols, vals = cols[sel], vals[sel]
        cols = cols[np.argsort(-vals)]
        i2i[auniq[a]] = [auniq[c] for c in cols]
    return i2i


def s4_i2i(hist, i2i_map, seeds=I2I_SEEDS):
    seed_arts = _recent_distinct(hist, seeds)
    out = {}
    for cust, arts in seed_arts.items():
        pool = []
        for a in arts:
            pool.extend(i2i_map.get(a, []))
        if pool:
            out[cust] = list(dict.fromkeys(pool))
    return out


def s5_index_group(hist, topn=IDX_TOP_N, k=RECENT_WEEKS):
    h = hist.merge(ART_META[['article_id', 'index_group_name']],
                   on='article_id', how='left')
    weeks = sorted(h['week'].unique())[-k:]
    recent = h[h['week'].isin(weeks)]
    best = (recent.groupby('index_group_name')['article_id']
            .apply(lambda s: s.value_counts().head(topn).index.tolist())
            .to_dict())
    fav = (h.groupby('customer_id')['index_group_name']
           .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else np.nan)
           .to_dict())
    return {c: list(best.get(g, [])) for c, g in fav.items() if g in best}


def s6_cf(hist, cust_list, weeks, factors=CF_FACTORS,
          iters=CF_ITERS, reg=CF_REG, topk=CF_TOPK):
    from implicit.als import AlternatingLeastSquares
    ws = sorted(hist['week'].unique())[-weeks:]
    d = hist[hist['week'].isin(ws)]
    ccodes, cuniq = pd.factorize(d['customer_id'])
    acodes, auniq = pd.factorize(d['article_id'])
    M = csr_matrix((np.ones(len(d), np.float32), (ccodes, acodes)),
                   shape=(len(cuniq), len(auniq)))
    M.data[:] = 1.0

    model = AlternatingLeastSquares(factors=factors, regularization=reg,
                                    iterations=iters, random_state=SEED)
    model.fit(M)
    uf = np.asarray(model.user_factors, dtype=np.float32)
    itf = np.asarray(model.item_factors, dtype=np.float32)

    pos = {c: i for i, c in enumerate(cuniq)}
    targets = [c for c in cust_list if c in pos]
    out, B = {}, 1000
    for i in range(0, len(targets), B):
        chunk = targets[i:i + B]
        rows = np.fromiter((pos[c] for c in chunk), dtype=np.int64, count=len(chunk))
        scores = uf[rows] @ itf.T
        part = np.argpartition(-scores, topk, axis=1)[:, :topk]
        for j, c in enumerate(chunk):
            cols = part[j]
            out[c] = [auniq[x] for x in cols[np.argsort(-scores[j, cols])]]
        del scores
    return out


# ------------------------- assemble candidate frame --------------------------
SOURCES = ['s1_recent', 's2_seasonal', 's3_repurchase', 's4_i2i', 's5_idx',
           's6_cf8', 's6_cf52']


def build_candidates(hist, cust_list, target_week):
    week_number = int(target_week.split('-')[1])
    srcs = {}
    t = time.time()
    srcs['s1_recent'] = s1_recent_bestsellers(hist, cust_list)
    srcs['s2_seasonal'] = s2_seasonal_bestsellers(hist, cust_list, week_number)
    srcs['s3_repurchase'] = s3_repurchase(hist)
    i2i_map = build_i2i_map(hist)
    srcs['s4_i2i'] = s4_i2i(hist, i2i_map)
    srcs['s5_idx'] = s5_index_group(hist)
    for w in CF_WINDOWS:
        key = f's6_cf{w}'
        if not USE_CF:
            srcs[key] = {}
            continue
        try:
            srcs[key] = s6_cf(hist, cust_list, weeks=w)
            print(f"  CF {w}wk: {len(srcs[key]):,} customers")
        except Exception as ex:
            print(f"  !! CF {w}wk FAILED: {type(ex).__name__}: {ex}")
            srcs[key] = {}
    print(f"  sources built in {time.time()-t:.0f}s")

    cust_set = set(cust_list)
    frames = []
    for name, d in srcs.items():
        pairs = [(c, a) for c, arts in d.items() if c in cust_set for a in arts]
        if not pairs:
            continue
        f = pd.DataFrame(pairs, columns=['customer_id', 'article_id'])
        f[name] = np.int8(1)
        frames.append(f)

    cand = frames[0]
    for f in frames[1:]:
        cand = cand.merge(f, on=['customer_id', 'article_id'], how='outer')
    for s in SOURCES:
        if s not in cand.columns:
            cand[s] = np.int8(0)
        cand[s] = cand[s].fillna(0).astype(np.int8)

    cand['n_sources'] = cand[SOURCES].sum(axis=1).astype(np.int8)
    if CANDIDATE_CAP is not None:
        cand = (cand.sort_values(['customer_id', 'n_sources'], ascending=[True, False])
                    .groupby('customer_id', sort=False).head(CANDIDATE_CAP))
    return cand.reset_index(drop=True)


# ------------------------------- features ------------------------------------
def add_features(cand, hist, customers, articles):
    cutoff = hist['t_dat'].max()
    recent_weeks = sorted(hist['week'].unique())[-RECENT_WEEKS:]
    recent = hist[hist['week'].isin(recent_weeks)]

    # ---- article level
    art = pd.DataFrame(index=hist['article_id'].unique())
    art.index.name = 'article_id'
    art['art_last_sale_days'] = (cutoff - hist.groupby('article_id')['t_dat'].max()).dt.days
    art['art_pop_all'] = hist['article_id'].value_counts()
    art['art_pop_recent'] = recent['article_id'].value_counts()
    art['art_mean_price'] = recent.groupby('article_id')['price'].mean()
    art['art_n_buyers_recent'] = recent.groupby('article_id')['customer_id'].nunique()
    art = art.fillna(0).reset_index()

    # ---- customer level
    cus = pd.DataFrame(index=hist['customer_id'].unique())
    cus.index.name = 'customer_id'
    cus['cust_n_tx'] = hist['customer_id'].value_counts()
    cus['cust_n_distinct_art'] = hist.groupby('customer_id')['article_id'].nunique()
    cus['cust_mean_price'] = hist.groupby('customer_id')['price'].mean()
    cus['cust_max_price'] = hist.groupby('customer_id')['price'].max()
    cus['cust_last_tx_days'] = (cutoff - hist.groupby('customer_id')['t_dat'].max()).dt.days
    cus = cus.reset_index()

    # ---- customer x article level
    pair = hist.groupby(['customer_id', 'article_id']).agg(
        pair_buy_count=('t_dat', 'size'),
        pair_last=('t_dat', 'max')).reset_index()
    pair['pair_last_days'] = (cutoff - pair['pair_last']).dt.days
    pair = pair.drop(columns='pair_last')

    # ---- colour / index affinity
    h = hist.merge(ART_META[['article_id', 'colour_group_name', 'index_group_name']],
                   on='article_id', how='left')
    tot = h.groupby('customer_id').size().rename('tot')
    col = h.groupby(['customer_id', 'colour_group_name']).size().rename('col_cnt').reset_index()
    col = col.merge(tot, on='customer_id')
    col['colour_share'] = col['col_cnt'] / col['tot']
    col = col[['customer_id', 'colour_group_name', 'colour_share']]

    idx = h.groupby(['customer_id', 'index_group_name']).size().rename('idx_cnt').reset_index()
    idx = idx.merge(tot, on='customer_id')
    idx['index_share'] = idx['idx_cnt'] / idx['tot']
    idx = idx[['customer_id', 'index_group_name', 'index_share']]
    del h; gc.collect()

    # ---- merge everything on
    out = cand.merge(ART_META[['article_id', 'colour_group_name', 'index_group_name']],
                     on='article_id', how='left')
    out = out.merge(art, on='article_id', how='left')
    out = out.merge(cus, on='customer_id', how='left')
    out = out.merge(pair, on=['customer_id', 'article_id'], how='left')
    out = out.merge(col, on=['customer_id', 'colour_group_name'], how='left')
    out = out.merge(idx, on=['customer_id', 'index_group_name'], how='left')
    out = out.merge(ART_META[['article_id', 'product_type_name',
                              'department_name', 'garment_group_name']],
                    on='article_id', how='left')
    out = out.merge(customers[['customer_id', 'age', 'club_member_status',
                               'fashion_news_frequency', 'Active', 'FN']],
                    on='customer_id', how='left')
    out['Active'] = out['Active'].fillna(0)
    out['FN'] = out['FN'].fillna(0)

    # ---- derived
    out['price_diff'] = out['art_mean_price'] - out['cust_mean_price']
    out['price_ratio'] = out['art_mean_price'] / out['cust_mean_price'].replace(0, np.nan)
    out['bought_before'] = out['pair_buy_count'].notna().astype(np.int8)

    out['pair_buy_count'] = out['pair_buy_count'].fillna(0)
    out['pair_last_days'] = out['pair_last_days'].fillna(999)
    out['colour_share'] = out['colour_share'].fillna(0)
    out['index_share'] = out['index_share'].fillna(0)

    out['matches_fav_category'] = 0  # filled by caller if fav map available

    for c in CAT_COLS:
        out[c] = out[c].astype('category')
    return out


FEATURES = SOURCES + [
    'n_sources',
    'art_last_sale_days', 'art_pop_all', 'art_pop_recent',
    'art_mean_price', 'art_n_buyers_recent',
    'cust_n_tx', 'cust_n_distinct_art', 'cust_mean_price',
    'cust_max_price', 'cust_last_tx_days',
    'pair_buy_count', 'pair_last_days', 'bought_before',
    'colour_share', 'index_share',
    'price_diff', 'price_ratio',
    'age', 'Active', 'FN',
] + CAT_COLS


# ------------------------------- labels / eval -------------------------------
def make_labels(cand, tx, target_week):
    truth = tx[tx['week'] == target_week][['customer_id', 'article_id']].drop_duplicates()
    truth['label'] = np.int8(1)
    out = cand.merge(truth, on=['customer_id', 'article_id'], how='left')
    out['label'] = out['label'].fillna(0).astype(np.int8)
    return out


def apk(actual, predicted, k=FINAL_K):
    predicted = predicted[:k]
    hits, score = 0, 0.0
    seen = set()
    for i, p in enumerate(predicted):
        if p in actual and p not in seen:
            hits += 1
            score += hits / (i + 1)
        seen.add(p)
    return score / min(len(actual), k)


def map_at_k(pred_dict, truth_dict, k=FINAL_K):
    return float(np.mean([apk(truth_dict[c], pred_dict.get(c, []), k)
                          for c in truth_dict]))


def rank_and_score(model, df, tx, target_week):
    df = df.copy()
    df['score'] = model.predict(df[FEATURES])
    df = df.sort_values(['customer_id', 'score'], ascending=[True, False])
    pred = (df.groupby('customer_id', sort=False)['article_id']
              .apply(lambda s: s.head(FINAL_K).tolist()).to_dict())
    t = tx[tx['week'] == target_week]
    truth = t.groupby('customer_id')['article_id'].apply(set).to_dict()
    return map_at_k(pred, truth), pred


# --------------------------- build one week's dataset ------------------------
def build_week(target_week, sample_customers=None, tag=""):
    print(f"\n=== {tag} week {target_week} ===")
    hist = tx[tx['week'] < target_week]
    buyers = tx.loc[tx['week'] == target_week, 'customer_id'].unique()
    if sample_customers is not None and len(buyers) > sample_customers:
        buyers = rng.choice(buyers, size=sample_customers, replace=False)
    print(f"  hist rows {len(hist):,} | customers scored {len(buyers):,}")

    cand = build_candidates(hist, list(buyers), target_week)
    print(f"  candidates {len(cand):,} "
          f"({len(cand)/max(len(buyers),1):.1f} per customer)")
    cand = make_labels(cand, tx, target_week)
    cand = add_features(cand, hist, customers, articles)
    pos = int(cand['label'].sum())
    print(f"  positives {pos:,} ({100*pos/len(cand):.3f}%)")

    truth = (tx[tx['week'] == target_week]
             .groupby('customer_id')['article_id'].apply(set))
    truth = truth[truth.index.isin(set(buyers))]
    recall = float(np.mean([
        len(truth[c] & set(g)) / len(truth[c])
        for c, g in cand.groupby('customer_id')['article_id'].apply(set).items()
        if c in truth.index]))
    print(f"  candidate recall {recall:.4f}")
    return cand


def fit_ranker(train_df, params):
    d = train_df.sort_values('customer_id').reset_index(drop=True)
    groups = d.groupby('customer_id', sort=False).size().values
    m = lgb.LGBMRanker(objective='lambdarank', metric='map',
                       random_state=SEED, n_jobs=-1, verbose=-1, **params)
    m.fit(d[FEATURES], d['label'], group=groups,
          categorical_feature=CAT_COLS)
    return m


# =============================== run =========================================
train_df = build_week(TRAIN_WEEK, sample_customers=TRAIN_CUST_SAMPLE, tag="TRAIN")
valid_df = build_week(VALID_WEEK, tag="VALID")

GRID = [
    dict(n_estimators=100, learning_rate=0.10, num_leaves=31,  min_child_samples=20),
    dict(n_estimators=300, learning_rate=0.05, num_leaves=63,  min_child_samples=50),
    dict(n_estimators=500, learning_rate=0.05, num_leaves=127, min_child_samples=100),
    dict(n_estimators=500, learning_rate=0.03, num_leaves=63,  min_child_samples=50,
         subsample=0.8, subsample_freq=1, colsample_bytree=0.8),
]

print("\n=== tuning on VALID week ===")
results = []
for i, p in enumerate(GRID):
    m = fit_ranker(train_df, p)
    score, _ = rank_and_score(m, valid_df, tx, VALID_WEEK)
    results.append((score, p))
    print(f"  [{i}] MAP@12 = {score:.5f}  {p}")

best_score, best_params = max(results, key=lambda x: x[0])
print(f"\nbest on valid: {best_score:.5f} -> {best_params}")

print("\nfeature importance (best config, trained on TRAIN week):")
best_model = fit_ranker(train_df, best_params)
imp = (pd.Series(best_model.feature_importances_, index=FEATURES)
       .sort_values(ascending=False))
print(imp.to_string())

# ---- final: retrain on the most recent labelled week, score the test week ONCE
print("\n=== FINAL: retrain on VALID week, evaluate on TEST week ===")
final_model = fit_ranker(valid_df, best_params)
test_df = build_week(TEST_WEEK, tag="TEST")
test_map, _ = rank_and_score(final_model, test_df, tx, TEST_WEEK)
print(f"\nTEST MAP@12 ({TEST_WEEK}) = {test_map:.5f}")
print(f"total runtime {(time.time()-t_start)/60:.1f} min")

loading...
  tx (31788324, 6) | customers (1371980, 7) | articles (105542, 25)

=== TRAIN week 2020-35 ===
  hist rows 30,916,085 | customers scored 76,757


/home/nbuser/.venv/lib/python3.10/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 40 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

  CF 8wk: 47,182 customers


  0%|          | 0/15 [00:00<?, ?it/s]

...


=== TEST week 2020-37 ===
  hist rows 31,452,491 | customers scored 67,790


  0%|          | 0/15 [00:00<?, ?it/s]

  CF 8wk: 42,151 customers


  0%|          | 0/15 [00:00<?, ?it/s]

  CF 52wk: 60,415 customers
  sources built in 653s
  candidates 12,170,101 (179.5 per customer)
  positives 29,611 (0.243%)
  candidate recall 0.1583

TEST MAP@12 (2020-37) = 0.03154
total runtime 60.5 min
